# Setup

Important: this notebook needs to be run with SageMath in **Python** mode.

In [1]:
from sage.all import (
    ZZ, RR, QQ, log, pi, e,
    vector, math, matrix,
    ceil, floor
)
    
from lattices import *

def LOG2(x):
    return RR(log(x, 2))

Set to True to print in the format of tables for latex:

In [2]:
print_latex_tables = False

This notebook mainly focuses on CROSS + reduced instance. While by default we have $z=7$, we consider hybrid attacks where we truncate $z$ to $z'\le7$ before doing the ListCVP/etc. This truncation cost is included in the computation. 
The modulus is always $p=127$.

In [3]:
p = 127
z_initial = 7

# Hybrid BatchCVP estimates

In [4]:
params = [
#    (20, 12),
    (35, 21),
#    (50, 30),
    (127, 76),
    (187, 111),
    (251, 150),
]
for n, k in params[:]:
    if print_latex_tables:
        print(r"\multirow{%d}{*}{(%d, %d)} " % (7, n, k))
    else:
        print("n", n, "k", k, "n-k", n-k)
    
    for z in reversed(range(2, 8)):
        # expected number of attempts
        cost_z = (QQ(z)/z_initial)**-n

        #n_sols = QQ(z)**n / 127**(n-k)

        # basic CVP cost
        bincvp = 2**(0.292*z*n)

        # BatchCVP amortized cost
        # number of guessed entries (to amortize BatchCVP)
        g = ceil(RR(0.058 * z * n * log(2, z)))
        assert z**g >= 2**(0.058*z*(n-g)), (z**g, 2**(0.058*z*(n-g)))
        bincvp2 = 2**(0.234*z*(n - g)) * z**g

        mem = 2**(0.208*z*(n-g))
        trunc_factor = 2.4  # increasing success rate of truncation to 90%
        if print_latex_tables:
            print(fr"& {z} & $2^{{{-LOG2(cost_z):.1f}}}$ "
                  fr"& {g} ($2^{{{LOG2(z**g):.1f}}}$)"
                  fr"& $2^{{{LOG2(bincvp2):.1f}}}$ "
                  fr"& $2^{{{LOG2(bincvp2 * cost_z * trunc_factor):.1f}}}$"
                  fr"& $2^{{{LOG2(mem):.1f}}}$ \\".replace("Infinity", r"\infty")
            )
        else:
            print(
                "z'", z, ":",
                "trunc.cost 2^%6.2f" % LOG2(cost_z),
                "guess %d^%d = 2^%6.2f" % (z, g, LOG2(z**g)),
                "BatchCVP cost 2^%6.2f" % LOG2(bincvp2),
                "total 2^%6.2f" % LOG2(cost_z * bincvp2 * trunc_factor),
            )
    if print_latex_tables:
        print(r"\midrule\n%")
    else:
        print()

n 35 k 21 n-k 14
z' 7 : trunc.cost 2^  0.00 guess 7^6 = 2^ 16.84 BatchCVP cost 2^ 64.35 total 2^ 65.61
z' 6 : trunc.cost 2^  7.78 guess 6^5 = 2^ 12.92 BatchCVP cost 2^ 55.04 total 2^ 64.09
z' 5 : trunc.cost 2^ 16.99 guess 5^5 = 2^ 11.61 BatchCVP cost 2^ 46.71 total 2^ 64.96
z' 4 : trunc.cost 2^ 28.26 guess 4^5 = 2^ 10.00 BatchCVP cost 2^ 38.08 total 2^ 67.60
z' 3 : trunc.cost 2^ 42.78 guess 3^4 = 2^  6.34 BatchCVP cost 2^ 28.10 total 2^ 72.15
z' 2 : trunc.cost 2^ 63.26 guess 2^5 = 2^  5.00 BatchCVP cost 2^ 19.04 total 2^ 83.56

n 127 k 76 n-k 51
z' 7 : trunc.cost 2^  0.00 guess 7^19 = 2^ 53.34 BatchCVP cost 2^230.24 total 2^231.51
z' 6 : trunc.cost 2^ 28.24 guess 6^18 = 2^ 46.53 BatchCVP cost 2^199.57 total 2^229.07
z' 5 : trunc.cost 2^ 61.65 guess 5^16 = 2^ 37.15 BatchCVP cost 2^167.02 total 2^229.93
z' 4 : trunc.cost 2^102.53 guess 4^15 = 2^ 30.00 BatchCVP cost 2^134.83 total 2^238.63
z' 3 : trunc.cost 2^155.24 guess 3^14 = 2^ 22.19 BatchCVP cost 2^101.52 total 2^258.02
z' 2 : trunc.

# Hybrid ListCVP estimates

In [5]:
for n, k in params:
    #delta0_hkz = RR((n/(2*pi*e))**(1/(2*n)))
    # more accurate:
    # lambda_1 ~= ball_vol(n)^-1/n  - GH
    # root Hermite factor = lambda_1^1/n
    # => 1/^n^2
    delta0_hkz = RR(ball_vol(n)**(-1/n**2))
    
    vol = 127**(n-k)
    sieve = 2**(0.292*n)
    
    if print_latex_tables:
        print(r"\multirow{%d}{*}{(%d, %d)} " % (7, n, k))
    else:
        print("n", n, "k", k, "n-k", n-k, f"sieve 2^{LOG2(sieve):5.2f}", "delta0 HKZ", delta0_hkz)

    for z in reversed(range(1, 8)):
        # expected number of attempts
        cost_z = (QQ(z)/z_initial)**-n
        
        #n_sols = QQ(z)**n / 127**(n-k)

        pts = [vector(QQ, [2**i]) for i in range(z)]
        mid = sum(pts) / len(pts)
        #mid[0] = round(mid[0])  # for ListSVP
        norm1_sqr = sum((v-mid).norm().n()**2 for v in pts) / len(pts)
        norm = norm1_sqr**0.5 * n**0.5
        enum_cost = 1 + enum_cost_delta(n, vol, norm, delta0_hkz, pruning=True)
        
        output_size = ball_vol(n, norm) / vol
        if output_size > 1:
            cost = sieve + 4 * enum_cost 
        else:
            cost = sieve

        mem = 2**(0.208*n)

        # increasing success rate of truncation + reduction
        trunc_factor = 4
        red_factor = 4 
        if print_latex_tables:
            print(fr"& {z} & $2^{{{-LOG2(cost_z):.1f}}}$ "
                  fr"& $2^{{{LOG2(output_size):.1f}}}$ "
                  fr"& $2^{{{LOG2(sieve):.1f}}}$ "
                  fr"& $2^{{{LOG2(enum_cost):.1f}}}$ "
                  fr"& $2^{{{LOG2(cost * cost_z * trunc_factor * red_factor):.1f}}}$"
                  fr"& $2^{{{LOG2(mem):.1f}}}$ \\".replace("Infinity", r"\infty")
            )
        else:
            if output_size <= 1:
                pts_str = "     < 1"
            else:
                pts_str = f"2^{LOG2(output_size):6.2f}"
            print(f"z' {z}"
                  f" trunc. cost: 2^{LOG2( cost_z):7.2f}",
                  f" #pts in BnR: {pts_str}",
                  f" enum cost 2^{LOG2(enum_cost):6.2f}",
                  f" total 2^{LOG2(cost * cost_z * trunc_factor * red_factor):7.2f}",
            )
    if print_latex_tables:
        print(r"\midrule\n%")
    else:
        print()

n 35 k 21 n-k 14 sieve 2^10.22 delta0 HKZ 1.01224625035208
z' 7 trunc. cost: 2^   0.00  #pts in BnR: 2^124.71  enum cost 2^130.02  total 2^ 136.02
z' 6 trunc. cost: 2^   7.78  #pts in BnR: 2^ 90.69  enum cost 2^ 96.22  total 2^ 110.00
z' 5 trunc. cost: 2^  16.99  #pts in BnR: 2^ 56.08  enum cost 2^ 62.12  total 2^  85.11
z' 4 trunc. cost: 2^  28.26  #pts in BnR: 2^ 20.21  enum cost 2^ 28.32  total 2^  62.58
z' 3 trunc. cost: 2^  42.78  #pts in BnR:      < 1  enum cost 2^  7.25  total 2^  57.00
z' 2 trunc. cost: 2^  63.26  #pts in BnR:      < 1  enum cost 2^  3.68  total 2^  77.48
z' 1 trunc. cost: 2^  98.26  #pts in BnR:      < 1  enum cost 2^  0.00  total 2^ 112.48

n 127 k 76 n-k 51 sieve 2^37.08 delta0 HKZ 1.00811736510622
z' 7 trunc. cost: 2^   0.00  #pts in BnR: 2^459.11  enum cost 2^466.46  total 2^ 472.46
z' 6 trunc. cost: 2^  28.24  #pts in BnR: 2^335.70  enum cost 2^343.51  total 2^ 377.75
z' 5 trunc. cost: 2^  61.65  #pts in BnR: 2^210.09  enum cost 2^219.55  total 2^ 287.20


# Hybrid ListSVP estimates

In [6]:
for n, k in params:    
    #delta0_hkz = RR((n/(2*pi*e))**(1/(2*n)))
    # more accurate:
    # lambda_1 ~= ball_vol(n)^-1/n  - GH
    # root Hermite factor = lambda_1^1/n
    # => 1/^n^2
    delta0_hkz = RR(ball_vol(n)**(-1/n**2))
    
    vol = 127**(n-k-1)  # reduced by 1 due to zeroizing the syndrome
    sieve = 2**(0.292*n)

    if print_latex_tables:
        print(r"\multirow{%d}{*}{(%d, %d)} " % (7, n, k))
    else:
        print("n", n, "k", k, "n-k", n-k, f"sieve 2^{LOG2(sieve):5.2f}", "delta0 HKZ", delta0_hkz)
    

    for z in reversed(range(1, 8)):
        # expected number of attempts
        cost_z = (QQ(z)/z_initial)**-n

        pts = [vector(QQ, [2**i]) for i in range(z)]
        mid = sum(pts) / len(pts)
        mid[0] = round(mid[0])  # for ListSVP
        
        norm1_sqr = sum((v-mid).norm().n()**2 for v in pts) / len(pts)
        norm = norm1_sqr**0.5 * n**0.5
        enum_cost = 1 + enum_cost_delta(n, vol, norm, delta0_hkz)
        
        output_size = ball_vol(n, norm) / vol
        if output_size > 10:
            cost = sieve + 4 * enum_cost 
        else:
            cost = sieve
        mem = 2**(0.208*n)

        # increasing success rate of truncation + reduction
        trunc_factor = 4
        red_factor = 4 
        if print_latex_tables:
            print(fr"& {z} & $2^{{{-RR(log(cost_z, 2)):.1f}}}$ "
                  fr"& $2^{{{RR(log(output_size, 2)):.1f}}}$ "
                  fr"& $2^{{{RR(log(sieve, 2)):.1f}}}$ "
                  fr"& $2^{{{RR(log(enum_cost, 2)):.1f}}}$ "
                  fr"& $2^{{{RR(log(cost * cost_z * trunc_factor * red_factor, 2)):.1f}}}$"
                  fr"& $2^{{{RR(log(mem, 2)):.1f}}}$ \\".replace("Infinity", r"\infty")
            )
        else:
            if output_size <= 1:
                pts_str = "     < 1"
            else:
                pts_str = f"2^{LOG2(output_size):6.2f}"
            print(f"z' {z}"
                  f" trunc. cost: 2^{LOG2( cost_z):7.2f}",
                  f" #pts in BnR: {pts_str}",
                  f" enum cost 2^{LOG2(enum_cost):6.2f}",
                  f" total 2^{LOG2(cost * cost_z * trunc_factor * red_factor):7.2f}",
            )
    if print_latex_tables:
        print(r"\midrule\n%")
    else:
        print()

n 35 k 21 n-k 14 sieve 2^10.22 delta0 HKZ 1.01224625035208
z' 7 trunc. cost: 2^   0.00  #pts in BnR: 2^131.70  enum cost 2^136.99  total 2^ 142.99
z' 6 trunc. cost: 2^   7.78  #pts in BnR: 2^ 97.74  enum cost 2^103.20  total 2^ 116.99
z' 5 trunc. cost: 2^  16.99  #pts in BnR: 2^ 63.10  enum cost 2^ 68.99  total 2^  91.98
z' 4 trunc. cost: 2^  28.26  #pts in BnR: 2^ 27.41  enum cost 2^ 34.77  total 2^  69.03
z' 3 trunc. cost: 2^  42.78  #pts in BnR:      < 1  enum cost 2^  9.37  total 2^  57.00
z' 2 trunc. cost: 2^  63.26  #pts in BnR:      < 1  enum cost 2^  4.93  total 2^  77.48
z' 1 trunc. cost: 2^  98.26  #pts in BnR:      < 1  enum cost 2^  0.00  total 2^ 112.48

n 127 k 76 n-k 51 sieve 2^37.08 delta0 HKZ 1.00811736510622
z' 7 trunc. cost: 2^   0.00  #pts in BnR: 2^466.10  enum cost 2^473.44  total 2^ 479.44
z' 6 trunc. cost: 2^  28.24  #pts in BnR: 2^342.88  enum cost 2^350.65  total 2^ 384.89
z' 5 trunc. cost: 2^  61.65  #pts in BnR: 2^217.20  enum cost 2^226.47  total 2^ 294.12
